# **BMW Sales Data(2010-2024)**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import Ridge
import warnings

In [2]:
bmw_data = pd.read_csv('../data/BMW_sales_data(2010-2024).csv')

In [3]:
display(bmw_data.head())

display(bmw_data.info())

display(bmw_data.describe())

display(bmw_data.isnull().sum())

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification
0,5 Series,2016,Asia,Red,Petrol,Manual,3.5,151748,98740,8300,High
1,i8,2013,North America,Red,Hybrid,Automatic,1.6,121671,79219,3428,Low
2,5 Series,2022,North America,Blue,Petrol,Automatic,4.5,10991,113265,6994,Low
3,X3,2024,Middle East,Blue,Petrol,Automatic,1.7,27255,60971,4047,Low
4,7 Series,2020,South America,Black,Diesel,Manual,2.1,122131,49898,3080,Low


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Model                 50000 non-null  object 
 1   Year                  50000 non-null  int64  
 2   Region                50000 non-null  object 
 3   Color                 50000 non-null  object 
 4   Fuel_Type             50000 non-null  object 
 5   Transmission          50000 non-null  object 
 6   Engine_Size_L         50000 non-null  float64
 7   Mileage_KM            50000 non-null  int64  
 8   Price_USD             50000 non-null  int64  
 9   Sales_Volume          50000 non-null  int64  
 10  Sales_Classification  50000 non-null  object 
dtypes: float64(1), int64(4), object(6)
memory usage: 4.2+ MB


None

,Year,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,2017.015700,3.247180,100307.203140,75034.600900,5067.514680
std,4.324459,1.009078,57941.509344,25998.248882,2856.767125
min,2010.000000,1.500000,3.000000,30000.000000,100.000000
25%,2013.000000,2.400000,50178.000000,52434.750000,2588.000000
50%,2017.000000,3.200000,100388.500000,75011.500000,5087.000000
75%,2021.000000,4.100000,150630.250000,97628.250000,7537.250000
max,2024.000000,5.000000,199996.000000,119998.000000,9999.000000


Model                   0
Year                    0
Region                  0
Color                   0
Fuel_Type               0
Transmission            0
Engine_Size_L           0
Mileage_KM              0
Price_USD               0
Sales_Volume            0
Sales_Classification    0
dtype: int64

In [4]:
#  Lets get an understanding of the trends in the data
sales_trend = bmw_data.groupby('Year')['Sales_Volume'].sum().reset_index()
fig = px.line(sales_trend, x='Year', y='Sales_Volume', title='BMW Sales Over Years')
fig.show()

# Analyzing sales by Model
model_sales = bmw_data.groupby('Model')['Sales_Volume'].sum().reset_index()
model_sales = model_sales.sort_values(by='Sales_Volume', ascending=False).head(10)
fig = px.bar(model_sales, x='Model', y='Sales_Volume', title='Top 10 BMW Models by Sales Volume', color_continuous_scale='Blues')
fig.show()

# Sales by Region
region_sales = bmw_data.groupby('Region')['Sales_Volume'].sum().reset_index()
fig = px.pie(
    region_sales,
    values='Sales_Volume',
    names='Region',    
    title='Proportion of BMW Sales by Region'
)
fig.show()

In [5]:
# Fuel Type Preferences Over Time
fuel_trend = bmw_data.groupby(['Year', 'Fuel_Type'])['Sales_Volume'].sum().reset_index()
fig = px.line(fuel_trend, x='Year', y='Sales_Volume', color='Fuel_Type', 
              title='Fuel Type Preferences Over Time',
              markers=True)
fig.update_layout(hovermode='x unified')
fig.show()

# Engine Size vs Sales Volume
engine_sales = bmw_data.groupby('Engine_Size_L')['Sales_Volume'].sum().reset_index()
fig = px.scatter(engine_sales, x='Engine_Size_L', y='Sales_Volume', 
                 size='Sales_Volume', color='Sales_Volume',
                 title='Engine Size vs Sales Volume',
                 color_continuous_scale='Viridis')
fig.show()

In [6]:
# Price vs Sales Volume by Model
price_model = bmw_data.groupby('Model').agg({'Price_USD': 'mean', 'Sales_Volume': 'sum'}).reset_index()
fig = px.scatter(price_model, x='Price_USD', y='Sales_Volume', 
                 size='Sales_Volume', hover_name='Model',
                 color='Sales_Volume', color_continuous_scale='Blues',
                 title='Price vs Sales Volume by Model')
fig.show()

# Transmission Type Distribution
transmission_sales = bmw_data.groupby('Transmission')['Sales_Volume'].sum().reset_index()
fig = px.bar(transmission_sales, x='Transmission', y='Sales_Volume',
             color='Transmission', title='Sales by Transmission Type',
             color_discrete_map={'Manual': '#1f77b4', 'Automatic': '#ff7f0e'})
fig.show()

In [7]:
# Color Popularity Analysis
color_sales = bmw_data.groupby('Color')['Sales_Volume'].sum().reset_index()
color_sales = color_sales.sort_values('Sales_Volume', ascending=False)
fig = px.bar(color_sales, x='Color', y='Sales_Volume',
             color='Sales_Volume', color_continuous_scale='Plasma',
             title='BMW Color Preferences by Sales Volume')
fig.show()

# Regional and Fuel Type Heatmap
region_fuel = bmw_data.pivot_table(values='Sales_Volume', index='Region', 
                                    columns='Fuel_Type', aggfunc='sum')
fig = go.Figure(data=go.Heatmap(z=region_fuel.values, 
                                 x=region_fuel.columns, 
                                 y=region_fuel.index,
                                 colorscale='YlOrRd'))
fig.update_layout(title='Sales Volume: Region vs Fuel Type Heatmap',
                  xaxis_title='Fuel Type', yaxis_title='Region')
fig.show()

In [8]:
# Sales Classification Distribution
sales_class = bmw_data.groupby('Sales_Classification')['Sales_Volume'].sum().reset_index()
fig = px.pie(sales_class, values='Sales_Volume', names='Sales_Classification',
             title='Sales Distribution by Classification (High vs Low)',
             color_discrete_sequence=['#2ecc71', '#e74c3c'])
fig.show()

# Average Price by Model and Fuel Type
price_model_fuel = bmw_data.groupby(['Model', 'Fuel_Type']).agg({
    'Price_USD': 'mean',
    'Sales_Volume': 'sum'
}).reset_index()

# Get top 8 models by sales
top_models = bmw_data.groupby('Model')['Sales_Volume'].sum().nlargest(8).index
price_model_fuel_filtered = price_model_fuel[price_model_fuel['Model'].isin(top_models)]

fig = px.bar(price_model_fuel_filtered, x='Model', y='Price_USD', 
             color='Fuel_Type', barmode='group',
             title='Average Price by Model and Fuel Type (Top 8 Models)',
             color_discrete_sequence=['#3498db', '#e74c3c', '#f39c12', '#9b59b6'])
fig.show()

In [9]:
# Year-over-Year Regional Performance
year_region_sales = bmw_data.groupby(['Year', 'Region'])['Sales_Volume'].sum().reset_index()
fig = px.bar(year_region_sales, x='Year', y='Sales_Volume', color='Region',
             title='Year-over-Year Sales by Region',
             barmode='stack')
fig.show()

# Mileage Impact on Sales (Depreciation Analysis)
mileage_bins = [0, 50000, 100000, 150000, 200000]
mileage_labels = ['0-50K', '50-100K', '100-150K', '150-200K']
bmw_data['Mileage_Category'] = pd.cut(bmw_data['Mileage_KM'], bins=mileage_bins, labels=mileage_labels)
mileage_sales = bmw_data.groupby('Mileage_Category')['Sales_Volume'].sum().reset_index()

fig = px.bar(mileage_sales, x='Mileage_Category', y='Sales_Volume',
             color='Sales_Volume', color_continuous_scale='RdYlGn_r',
             title='Sales Volume by Vehicle Mileage Range')
fig.show()

/tmp/ipykernel_40040/281327956.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [10]:
# Key Insights Summary
print("=" * 70)
print("BMW SALES INSIGHTS & TRENDS (2010-2024)")
print("=" * 70)

# 1. Fuel Type Trend
petrol_pct = (bmw_data[bmw_data['Fuel_Type'] == 'Petrol']['Sales_Volume'].sum() / 
              bmw_data['Sales_Volume'].sum() * 100)
diesel_pct = (bmw_data[bmw_data['Fuel_Type'] == 'Diesel']['Sales_Volume'].sum() / 
              bmw_data['Sales_Volume'].sum() * 100)
hybrid_pct = (bmw_data[bmw_data['Fuel_Type'] == 'Hybrid']['Sales_Volume'].sum() / 
              bmw_data['Sales_Volume'].sum() * 100)

print(f"\n1. FUEL TYPE DOMINANCE:")
print(f"   • Petrol: {petrol_pct:.1f}% of total sales")
print(f"   • Diesel: {diesel_pct:.1f}% of total sales")
print(f"   • Hybrid: {hybrid_pct:.1f}% of total sales")

# 2. Top performing models
top_3_models = bmw_data.groupby('Model')['Sales_Volume'].sum().nlargest(3)
print(f"\n2. TOP PERFORMING MODELS:")
for idx, (model, sales) in enumerate(top_3_models.items(), 1):
    print(f"   {idx}. {model}: {sales:,} units sold")

# 3. Regional insights
top_region = bmw_data.groupby('Region')['Sales_Volume'].sum().idxmax()
region_sales_max = bmw_data.groupby('Region')['Sales_Volume'].sum().max()
print(f"\n3. REGIONAL LEADER:")
print(f"   • {top_region}: {region_sales_max:,} units ({region_sales_max / bmw_data['Sales_Volume'].sum() * 100:.1f}% market share)")

# 4. Color preferences
top_color = bmw_data.groupby('Color')['Sales_Volume'].sum().idxmax()
color_sales_max = bmw_data.groupby('Color')['Sales_Volume'].sum().max()
print(f"\n4. PREFERRED COLOR:")
print(f"   • {top_color}: {color_sales_max:,} units sold")

# 5. Transmission preference
auto_sales = bmw_data[bmw_data['Transmission'] == 'Automatic']['Sales_Volume'].sum()
manual_sales = bmw_data[bmw_data['Transmission'] == 'Manual']['Sales_Volume'].sum()
print(f"\n5. TRANSMISSION PREFERENCE:")
print(f"   • Automatic: {auto_sales / (auto_sales + manual_sales) * 100:.1f}%")
print(f"   • Manual: {manual_sales / (auto_sales + manual_sales) * 100:.1f}%")

# 6. Price insights
avg_price = bmw_data['Price_USD'].mean()
print(f"\n6. PRICING INSIGHTS:")
print(f"   • Average BMW Price: ${avg_price:,.2f}")
print(f"   • Price Range: ${bmw_data['Price_USD'].min():,.0f} - ${bmw_data['Price_USD'].max():,.0f}")

# 7. Sales classification
high_sales_pct = (bmw_data[bmw_data['Sales_Classification'] == 'High']['Sales_Volume'].sum() / 
                  bmw_data['Sales_Volume'].sum() * 100)
print(f"\n7. SALES CLASSIFICATION:")
print(f"   • High-Sales Models: {high_sales_pct:.1f}% of total sales volume")

# 8. Year trend
recent_year_sales = bmw_data[bmw_data['Year'] >= 2022]['Sales_Volume'].sum()
early_year_sales = bmw_data[bmw_data['Year'] <= 2015]['Sales_Volume'].sum()
print(f"\n8. TEMPORAL TREND:")
print(f"   • Recent years (2022-2024): {recent_year_sales:,} units")
print(f"   • Early years (2010-2015): {early_year_sales:,} units")

print("\n" + "=" * 70)

BMW SALES INSIGHTS & TRENDS (2010-2024)

1. FUEL TYPE DOMINANCE:
   • Petrol: 25.0% of total sales
   • Diesel: 24.6% of total sales
   • Hybrid: 25.5% of total sales

2. TOP PERFORMING MODELS:
   1. 7 Series: 23,786,466 units sold
   2. i8: 23,423,891 units sold
   3. X1: 23,406,060 units sold

3. REGIONAL LEADER:
   • Asia: 42,974,277 units (17.0% market share)

4. PREFERRED COLOR:
   • Red: 42,750,183 units sold

5. TRANSMISSION PREFERENCE:
   • Automatic: 49.7%
   • Manual: 50.3%

6. PRICING INSIGHTS:
   • Average BMW Price: $75,034.60
   • Price Range: $30,000 - $119,998

7. SALES CLASSIFICATION:
   • High-Sales Models: 51.1% of total sales volume

8. TEMPORAL TREND:
   • Recent years (2022-2024): 51,717,454 units
   • Early years (2010-2015): 101,280,181 units



___Price Prediction Model___

In [11]:
# We can create a price prediction model based on features like Model, Year, Mileage, Fuel Type, and Transmission.

bmw_train_data = bmw_data[['Model', 'Year', 'Mileage_KM', 'Fuel_Type', 'Transmission', 'Engine_Size_L', 'Price_USD']].dropna()

In [ ]:
warnings.filterwarnings('ignore')

bmw_train_data.info()

bmw_encoded_data = bmw_train_data.copy()
# lets label encode categorical features and prepare for modeling
le = LabelEncoder()
bmw_encoded_data['Transmission'] = le.fit_transform(bmw_train_data['Transmission'])

# for the categorical features with more than 2 categories, we will use one hot encoding
bmw_encoded_data = pd.get_dummies(bmw_encoded_data, columns=['Model', 'Fuel_Type'], drop_first=True)

# Feature scaling for better model performance
scaler = StandardScaler()
feature_columns = [col for col in bmw_encoded_data.columns if col != 'Price_USD']
bmw_encoded_data[feature_columns] = scaler.fit_transform(bmw_encoded_data[feature_columns])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Model          50000 non-null  object 
 1   Year           50000 non-null  int64  
 2   Mileage_KM     50000 non-null  int64  
 3   Fuel_Type      50000 non-null  object 
 4   Transmission   50000 non-null  object 
 5   Engine_Size_L  50000 non-null  float64
 6   Price_USD      50000 non-null  int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 2.7+ MB


In [14]:
train_data, test_data = train_test_split(bmw_encoded_data, test_size=0.2, random_state=42)

train_data_X = train_data.drop('Price_USD', axis=1)
train_data_y = train_data['Price_USD']

test_data_X = test_data.drop('Price_USD', axis=1)
test_data_y = test_data['Price_USD']

In [ ]:
# Train AdaBoost Model (Best Performer)

BestModel = AdaBoostRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)
BestModel.fit(train_data_X, train_data_y)

Training AdaBoost Regressor...
✓ AdaBoost model trained successfully!
✓ AdaBoost model trained successfully!


In [45]:
# Generate predictions using AdaBoost
predictions = BestModel.predict(test_data_X)
mae = mean_absolute_error(test_data_y, predictions)
mse = mean_squared_error(test_data_y, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(test_data_y, predictions)

In [ ]:
# Display Model Performance
print("ADABOOST PRICE PREDICTION MODEL - FINAL PERFORMANCE")
print("-" * 80)
print(f"\nModel: AdaBoost Regressor")
print(f"Number of Estimators: 200")
print(f"Learning Rate: 0.05")
print(f"\nTraining Set Size: {len(train_data_X)} samples")
print(f"Testing Set Size: {len(test_data_X)} samples")
print("\n" + "-" * 80)
print(f"Performance Metrics:")
print(f"  • Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"  • Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"  • R² Score: {r2:.4f}")
print("-" * 80)

ADABOOST PRICE PREDICTION MODEL - FINAL PERFORMANCE
--------------------------------------------------------------------------------

Model: AdaBoost Regressor
Number of Estimators: 200
Learning Rate: 0.05

Training Set Size: 40000 samples
Testing Set Size: 10000 samples

--------------------------------------------------------------------------------
Performance Metrics:
  • Mean Absolute Error (MAE): $22,563.19
  • Root Mean Squared Error (RMSE): $26,018.64
  • R² Score: -0.0001
--------------------------------------------------------------------------------



In [51]:
# Model Dumping and Serialization
import joblib
import os

# Create models directory if it doesn't exist
models_dir = '../models'
if not os.path.exists(models_dir):
    os.makedirs(models_dir)

# Save the trained AdaBoost model
model_path = os.path.join(models_dir, 'bmw_adaboost_price_model.joblib')
joblib.dump(BestModel, model_path)

# Save the label encoder
le_path = os.path.join(models_dir, 'bmw_transmission_encoder.joblib')
joblib.dump(le, le_path)

# Save feature columns for future predictions
feature_cols_path = os.path.join(models_dir, 'bmw_feature_columns.joblib')
joblib.dump(train_data_X.columns.tolist(), feature_cols_path)

['../models/bmw_feature_columns.joblib']

In [ ]:
# Prediction Function for New Data
def predict_bmw_price(model_name, year, mileage_km, fuel_type, transmission, engine_size_l):
    """
    Predict BMW price using the trained AdaBoost model.
    
    Parameters:
    -----------
    model_name : str
        BMW model (e.g., '3 Series', '5 Series', 'X3', 'X5', '7 Series', 'i8')
    year : int
        Year of the vehicle (2010-2024)
    mileage_km : float
        Mileage in kilometers
    fuel_type : str
        Fuel type ('Petrol', 'Diesel', 'Hybrid')
    transmission : str
        Transmission type ('Manual', 'Automatic')
    engine_size_l : float
        Engine size in liters
    
    Returns:
    --------
    float : Predicted price in USD
    """
    
    # Create a dataframe with the input
    input_data = pd.DataFrame({
        'Model': [model_name],
        'Year': [year],
        'Mileage_KM': [mileage_km],
        'Fuel_Type': [fuel_type],
        'Transmission': [transmission],
        'Engine_Size_L': [engine_size_l]
    })
    
    # Encode transmission
    input_data['Transmission'] = le.transform(input_data['Transmission'])
    
    # One-hot encode categorical features
    input_data = pd.get_dummies(input_data, columns=['Model', 'Fuel_Type'], drop_first=True)
    
    # Ensure all required columns are present
    for col in train_data_X.columns:
        if col not in input_data.columns:
            input_data[col] = 0
    
    # Reorder columns to match training data
    input_data = input_data[train_data_X.columns]
    
    # Scale the features
    input_data_scaled = scaler.transform(input_data)
    
    # Make prediction
    predicted_price = BestModel.predict(input_data_scaled)[0]
    
    return predicted_price

# Test the prediction function with sample data
print("\n" + "=" * 80)
print("TESTING PREDICTION FUNCTION - SAMPLE PREDICTIONS")
print("=" * 80)

test_cases = [
    {"model": "3 Series", "year": 2022, "mileage": 45000, "fuel": "Petrol", "trans": "Automatic", "engine": 2.0},
    {"model": "5 Series", "year": 2020, "mileage": 80000, "fuel": "Diesel", "trans": "Automatic", "engine": 3.0},
    {"model": "X3", "year": 2023, "mileage": 20000, "fuel": "Petrol", "trans": "Automatic", "engine": 2.5},
    {"model": "X5", "year": 2019, "mileage": 120000, "fuel": "Diesel", "trans": "Automatic", "engine": 3.5},
]

for i, case in enumerate(test_cases, 1):
    pred_price = predict_bmw_price(
        case["model"], case["year"], case["mileage"],
        case["fuel"], case["trans"], case["engine"]
    )
    print(f"\nSample {i}:")
    print(f"  Model: {case['model']} | Year: {case['year']} | Mileage: {case['mileage']:,} km")
    print(f"  Fuel: {case['fuel']} | Transmission: {case['trans']} | Engine: {case['engine']}L")
    print(f"  → Predicted Price: ${pred_price:,.2f}")

print("\n" + "=" * 80)